# 区别Detach和Data之间的具体细节

In [14]:
import torch

In [15]:
# 用两组数据进行对比
x1 = torch.tensor([1.0 ,2, 3],requires_grad=True)
x2 = torch.tensor([1.0 ,2, 3],requires_grad=True)
print(x1)
print(x2)

tensor([1., 2., 3.], requires_grad=True)
tensor([1., 2., 3.], requires_grad=True)


In [16]:
y1 = x1.sigmoid()
y2 = x2.sigmoid()
print(y1)
print(y2)


tensor([0.7311, 0.8808, 0.9526], grad_fn=<SigmoidBackward0>)
tensor([0.7311, 0.8808, 0.9526], grad_fn=<SigmoidBackward0>)


In [17]:
y1.sum().backward()
print(x1.grad)

tensor([0.1966, 0.1050, 0.0452])


In [18]:
y2.sum().backward()
print(x2.grad)

tensor([0.1966, 0.1050, 0.0452])


In [19]:
z1 = y1.data
z2 = y2.detach()
print(z1.requires_grad)
print(z2.requires_grad)

False
False


In [20]:
# ==========================================
# 区别 1：共享存储 —— 两者都和原 tensor 共享底层数据
# ==========================================
y1 = x1.sigmoid()
d1 = y1.data
d2 = y1.detach()

print("d1 和 y1 共享存储:", d1.data_ptr() == y1.data_ptr())
print("d2 和 y1 共享存储:", d2.data_ptr() == y1.data_ptr())


d1 和 y1 共享存储: True
d2 和 y1 共享存储: True


In [21]:
# ==========================================
# 区别 2：.data 谎称自己是叶子节点（is_leaf=True）
# 这在调试时会误导你，而 .detach() 如实地报告 False
# ==========================================
y = x1.sigmoid()

print("y.data.is_leaf:      ", y.data.is_leaf)      # True  ← 谎报！
print("y.detach().is_leaf:  ", y.detach().is_leaf)   # False ← 正确
print("y.is_leaf:           ", y.is_leaf)             # False
# y 来自 x1.sigmoid()，显然不是叶子节点，.data 却说是 True

y.data.is_leaf:       True
y.detach().is_leaf:   True
y.is_leaf:            False


In [22]:
# ==========================================
# 区别 3（核心危险！）：通过 .data 可以静默修改有梯度的 tensor
# 因为 .data 返回 requires_grad=False，in-place 操作被允许
# 但它和原 tensor 共享存储 → 原 tensor 被偷偷改掉了！
# ==========================================
x = torch.tensor([1.0, 2.0, 3.0], requires_grad=True)
y = x * 2

print("=== 修改前 ===")
print("y:", y)

# ⚠️ 危险操作：通过 .data 做 in-place 修改
y.data += 10   # 等价于 y.data.add_(10)，in-place！

print("\n=== 修改 y.data 后 ===")
print("y（被静默修改了！）:", y)

# 现在 backward 会报错，因为 autograd 检测到了 in-place 修改
try:
    y.sum().backward()
except RuntimeError as e:
    print("\n❌ backward 失败！", str(e).split(".")[0])
    print("   原因：y 被 in-place 修改，autograd 的版本检测发现数据变了")

=== 修改前 ===
y: tensor([2., 4., 6.], grad_fn=<MulBackward0>)

=== 修改 y.data 后 ===
y（被静默修改了！）: tensor([12., 14., 16.], grad_fn=<MulBackward0>)


In [23]:
# ==========================================
# 正确做法：用 .detach().clone() 获取独立副本
# .detach() 断开计算图，.clone() 复制存储
# 这样可以安全地修改数据而不影响原 tensor
# ==========================================
x = torch.tensor([1.0, 2.0, 3.0], requires_grad=True)
y = x.sigmoid()

# ✅ 正确：detach + clone = 完全独立的副本
z_safe = y.detach().clone()
print("z_safe 和 y 共享存储:", z_safe.data_ptr() == y.data_ptr())  # False！

z_safe += 100  # 安全修改，不影响 y
print("修改 z_safe 后 y 不变:", y)

# backward 正常
y.sum().backward()
print("梯度正常:", x.grad)

z_safe 和 y 共享存储: False
修改 z_safe 后 y 不变: tensor([0.7311, 0.8808, 0.9526], grad_fn=<SigmoidBackward0>)
梯度正常: tensor([0.1966, 0.1050, 0.0452])


In [24]:
# ==========================================
# 历史背景：为什么 .data 曾广泛使用？
# 早期 PyTorch 中，手动更新参数常用 .data 绕过 autograd
# 现代 PyTorch 中应使用 torch.no_grad() 或 optimizer.step()
# ==========================================
w = torch.tensor([2.0], requires_grad=True)

# ❌ 旧式写法（危险，别用）：
w.data -= 0.1 * w.grad  if w.grad is not None else 0  # 绕过了 autograd

# ✅ 现代写法：
with torch.no_grad():
    w -= 0.1 * (w.grad if w.grad is not None else 0)
print("权重更新完毕，w:", w)

权重更新完毕，w: tensor([2.], requires_grad=True)


## 总结

| 对比维度 | `.data` | `.detach()` |
|---|---|---|
| `requires_grad` | `False` | `False` |
| 共享存储 | ✅ 共享 | ✅ 共享 |
| `is_leaf` | 错误地返回 `True` | 正确返回 `False` |
| in-place 修改 | 允许（危险！会静默修改原 tensor） | 也共享存储，同样需小心 |
| 安全用法 | **不推荐使用** | `.detach().clone()` 获取独立副本 |
| 官方态度 | 历史遗留，不建议使用 | 推荐使用 |

**核心规则：永远用 `.detach()` 替代 `.data`，需要独立数据时使用 `.detach().clone()`。**